# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD):

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields (columns), and their `@id`s.


In [ ]:
print("Available Record Sets (by @id):")
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}, description: {rs.description}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      * @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from selected record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all available record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]

# Load all records from each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")

# For demonstration, select the main survey record set (replace below with actual @id from step 2 if needed)
# For this dataset, there is likely one main record set, get its @id:
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())

# Show a sample of the data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. This step uses field `@id`s where possible.

In [ ]:
import numpy as np

# Inspect the DataFrame and choose a numeric field (e.g., PHQ-9 score)
# Find a field id that looks like a numeric mental health score (use the 'phq_9_score' example, replace as needed)
# Normally, field @id would look like 'phq_9_score' or similar; adjust if different from actual exploration
numeric_field = None
for col in dataframes[main_record_set_id].columns:
    if 'phq' in col.lower() or 'gad' in col.lower() or 'psq' in col.lower():
        numeric_field = col
        break

if numeric_field:
    print(f"Selected numeric field: {numeric_field}")

    # Filter: keep respondents with PHQ-9 score greater than 10 (threshold for moderate depression)
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (N={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered respondents (mean=0, std=1):")
    display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

    # Optionally: group by an attribute, e.g., 'level_of_education' (if present)
    group_field = None
    for col in dataframes[main_record_set_id].columns:
        if 'educ' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field matching PHQ/GAD/PSQ found for EDA.")

## 5. Visualization
Visualize the distribution of the selected score field and relationships (e.g., boxplot by education level).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field], kde=True, bins=12, color='purple')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id], palette='Set2')
        plt.title(f"{numeric_field} Scores by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded Croissant metadata and survey records with `mlcroissant`
- Reviewed and explored available record sets, fields, and their `@id`s
- Extracted the main survey data, performed simple filtering and normalization on the PHQ-9 score (or similar field by `@id`), and grouped by education level
- Visualized the selected score field overall and by education level

This notebook can be further extended by referencing other record sets, using additional field `@id`s, and performing domain-specific analyses relevant to mental health research or public health policy in Kenya.